# Optimal-decay MCBFs in relative joint angles

Companion notebook to *Optimal Decay Matrix Control Barrier Functions: Soundness and
Completeness for Matrix Safety Constraints* ([arXiv:2610.18675](https://arxiv.org/abs/2610.18675)).
This version of the example uses relative joint angles and is not in the paper; `main.ipynb`
has the absolute-angle version the paper uses.

**Running it.** Set up the environment as in the README, start Jupyter from the repository
root (`jupyter lab`), and run the cells in order. The closed loop takes under a minute. The
last cell saves the animation as an mp4 if [FFmpeg](https://ffmpeg.org) is installed and
skips it otherwise.

The coordinates are $q=(q_1,q_2)$: base angle and elbow angle relative to the first link.
The physical link angles are $(q_1,q_1+q_2)$. Both the manipulability matrix $JJ^\top$ and
the control cost use relative joint rates $\dot q$, a different velocity metric from the
absolute-angle example. The arm model is `odmcbf/two_link_relative.py`. The rollouts store
the state in `Trace.theta`; here that array holds the **relative coordinates $q$**.

In [ ]:
import warnings

import numpy as np
import cvxpy as cp
from IPython.display import HTML

from odmcbf import SafetyFilter, run_comparison, summarize
from odmcbf import figures
from odmcbf.two_link_relative import RelativeTwoLinkArm, SIGMA_BAR_SQ, Q2_BOUNDARY

# Closed-loop setup, in relative joint rates.
GAMMA   = 1.0
OMEGA_D = 1.0
RHO     = 5e-3                     # rho_Omega = (4/2) * RHO = 1e-2
U_NOM   = np.array([1.0, -0.6])    # base rate and relative elbow rate [rad/s]
Q0      = np.array([0.0, 2*np.pi/3])
DT, HORIZON = 0.01, 8.0

arm = RelativeTwoLinkArm(gamma=GAMMA, sigma_bar_sq=SIGMA_BAR_SQ)
print(arm)
warnings.filterwarnings("ignore", message="Solution may be inaccurate", category=UserWarning)
print("solver:", "MOSEK" if "MOSEK" in cp.installed_solvers() else "CLARABEL",
      "| cvxpy", cp.__version__, "| numpy", np.__version__)


## 1. The constraint

The arm has unit links, configuration $\mathbf q=(q_1,q_2)$ and velocity control
$\dot{\mathbf q}=\mathbf u$. With Jacobian $\mathbf J(\mathbf q)$ and
$\mathbf M=\mathbf J\mathbf J^\top$, the matrix constraint

$$\mathbf H(\mathbf q)=\mathbf M(\mathbf q)-\bar\sigma^2\mathbf I\succeq\mathbf 0
\iff \sigma_{\min}(\mathbf J(\mathbf q))\ge\bar\sigma$$

keeps the arm away from singular configurations. The threshold
$\bar\sigma^2=(3-\sqrt5)/2$ puts the active boundary at $q_2=\pi/2$.

The base joint acts on $\mathbf H$ by rigid rotation, so the eigenvalues depend on $q_2$
alone: **base rotation on its own never changes the safety margin.**

In [ ]:
print(f"sigma_bar^2 = {arm.sigma_bar_sq:.6f}")
for q1 in (0.0, 0.55, 1.30):                     # the spectrum is independent of q1
    print(f"  eig H(q1={q1:+.2f}, pi/2) = {np.linalg.eigvalsh(arm.H([q1, Q2_BOUNDARY]))}")

step = 1e-6
slope = (arm.margin([0.4, Q2_BOUNDARY + step]) - arm.margin([0.4, Q2_BOUNDARY - step])) / (2*step)
print(f"\nd lambda_min(H) / d q2 at the boundary = {slope:.6f}  (= 3/sqrt(5) - 1 = {3/np.sqrt(5)-1:.6f})")
print("so the tangency condition at the boundary is u2 >= 0, with u1 unrestricted:")
print(f"the control closest to ({U_NOM[0]:g}, {U_NOM[1]:g}) satisfying it is (1.0, 0.0)")

## 2. Closed-loop comparison

Starting from $\mathbf q(0)=(0,2\pi/3)$, the commanded rate
$\mathbf k_{\rm nom}=(1,-0.6)$ rad/s is applied for 8 s. The elbow runs into the
boundary $q_2=\pi/2$ before reaching the straight-arm singularity.

A minimally invasive filter should brake elbow extension and leave the base rate alone.

In [ ]:
traces = run_comparison(arm, u_nom=U_NOM, theta0=Q0, dt=DT, horizon=HORIZON,
                        omega_d=OMEGA_D, rho_omega=RHO, recovery=True)
summary = {name: summarize(trace, U_NOM, OMEGA_D) for name, trace in traces.items()}

header = f"{'controller':<16}{'min margin':>12}{'q1(T) [rad]':>13}{'late u1':>10}{'decay dev':>11}{'solve [ms]':>12}"
print(header); print("-"*len(header))
for name, trace in traces.items():
    s = summary[name]
    fmt = lambda x, w, p=2: f"{x:>{w}.{p}f}" if x is not None else f"{'-':>{w}}"
    print(f"{trace.label:<16}{s['min_margin']:>12.2e}{s['joint1_rotation']:>13.2f}"
          f"{s['late_joint1_rate']:>10.3f}{fmt(s['decay_deviation_settled'],11)}"
          f"{fmt(s['mean_solve_ms'],12)}")
print("\nagainst the commanded 8.00 rad of base rotation; per-step solve times are machine dependent")

## 3. Animation

Each panel shows the arm, the singular-value ellipse at the end-effector, and the dashed
threshold circle. The constraint is active exactly when the ellipse's minor axis touches
the circle.

In [ ]:
HTML(figures.animate(traces, arm=arm, skip=8).to_jshtml(fps=12, default_mode="loop"))

## 4. Figures

The snapshot spacing is set per controller so that the different amounts of base rotation
are readable.

In [ ]:
# Per-controller snapshot spacing for this figure; restored afterwards.
previous_spacing = figures.SNAPSHOT_SPACING.copy()
try:
    figures.SNAPSHOT_SPACING.update(
        nominal=1.0, standard=2.0, scalar=1.6, matrix=1.0
    )
    relative_snapshots = figures.snapshots(traces, arm=arm)
finally:
    figures.SNAPSHOT_SPACING.clear()
    figures.SNAPSHOT_SPACING.update(previous_spacing)
relative_snapshots

In [ ]:
figures.closed_loop(traces, U_NOM, omega_d=OMEGA_D);     # margin, control and decay deviations

In [ ]:
# Optional: save the animation as an mp4 in figures/ (needs FFmpeg; skipped without it).
from pathlib import Path
from matplotlib.animation import FFMpegWriter

if FFMpegWriter.isAvailable():
    export_path = Path("figures") / "two_link_velocity_tracking.mp4"
    export_path.parent.mkdir(exist_ok=True)
    ani = figures.animate(traces, arm=arm, skip=8)
    writer = FFMpegWriter(fps=12, codec="libx264", extra_args=["-crf", "18", "-pix_fmt", "yuv420p"])
    ani.save(str(export_path), writer=writer, dpi=300)    # about 4080 x 1200 pixels
    print(f"Saved animation to {export_path.resolve()}")
else:
    print("FFmpeg not found, so the mp4 was not saved; the animation above is unaffected.")